# Corpus statistics — what's in the wisdom corpus?

This notebook reads the local public-domain source files in `assets/corpus/gold/` (and optionally the silver-tier `Abirate/english_quotes` jsonl) and answers four basic questions about what gets retrieved at generation time:

1. How many passages do we have per tradition?
2. What's the length distribution per tradition (short / medium / long buckets)?
3. What's the theme-tag coverage (which themes are over- and under-represented)?
4. Are any super-arms `(tradition, tone)` so sparse that the bandit's `min_count=3` filter would exclude them?

**Why local files instead of querying Supabase?** The local `assets/corpus/gold/*.txt` files are the ground truth — Supabase is just a derived index. Running this notebook against the local files means anyone (a recruiter cloning the repo) can render the plots without DB credentials, and the numbers always match what's actually deployed.

The chunking + tagging logic mirrors `scripts/seed_passages_to_supabase.py`. If that script changes, this notebook should be re-run.


In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Resolve the repo root regardless of where Jupyter was launched from.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    ROOT = NOTEBOOK_DIR.parent
else:
    ROOT = NOTEBOOK_DIR
GOLD_DIR = ROOT / "assets" / "corpus" / "gold"
SILVER_FILE = ROOT / "assets" / "corpus" / "silver" / "abirate_english_quotes.jsonl"
print(f"Repo root: {ROOT}")
print(f"Gold dir exists: {GOLD_DIR.exists()}")
print(f"Silver file exists: {SILVER_FILE.exists()}")

In [ ]:
# Mirror of scripts/seed_passages_to_supabase.py — keep these in sync.

GOLD_SOURCES: dict[str, tuple[str, str]] = {
    "bhagavad_gita_song_celestial.txt": ("bhagavad_gita", "Bhagavad Gita (Song Celestial)"),
    "tao_te_ching_legge.txt": ("tao_te_ching", "Tao Te Ching (Legge)"),
    "analects_legge.txt": ("analects", "Analects (Legge)"),
    "meditations_marcus_aurelius.txt": ("marcus_aurelius", "Meditations (Marcus Aurelius)"),
    "epictetus_discourses.txt": ("epictetus", "Discourses (Epictetus)"),
    "walden_thoreau.txt": ("thoreau", "Walden (Thoreau)"),
    "emerson_essays_first_series.txt": ("emerson", "Essays First Series (Emerson)"),
    "gibran_the_prophet.txt": ("gibran", "The Prophet (Gibran)"),
    "aesops_fables.txt": ("aesop", "Aesop's Fables"),
    "tagore_gitanjali.txt": ("tagore", "Gitanjali (Tagore)"),
    "dhammapada_muller.txt": ("dhammapada", "Dhammapada (Muller)"),
    "my_first_summer_in_the_sierra_muir.txt": ("muir", "My First Summer in the Sierra (Muir)"),
    "wake_robin_burroughs.txt": ("burroughs", "Wake-Robin (Burroughs)"),
}

KEYWORD_TAGS: dict[str, set[str]] = {
    "nature": {"nature", "forest", "river", "sky", "tree", "wind", "earth", "mountain", "bird"},
    "work": {"work", "labor", "craft", "duty", "effort", "task"},
    "presence": {"present", "today", "now", "attention", "aware", "mindful"},
    "courage": {"courage", "brave", "fear", "bold"},
    "impermanence": {"impermanence", "change", "passing", "death", "mortal"},
    "love": {"love", "heart", "friend", "friendship"},
    "discipline": {"discipline", "habit", "practice", "routine"},
    "joy": {"joy", "happiness", "delight"},
    "sorrow": {"sorrow", "grief", "pain"},
    "cosmos": {"universe", "infinite", "stars", "cosmos"},
    "animals": {"fox", "lion", "wolf", "bird", "animal", "dog", "cat"},
    "change": {"spring", "summer", "autumn", "fall", "winter", "season", "monsoon", "rain"},
}


def normalize_whitespace(text: str) -> str:
    text = text.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(raw: str) -> list[str]:
    cleaned = normalize_whitespace(raw)
    chunks = [c.strip() for c in re.split(r"\n\s*\n", cleaned) if c.strip()]
    return [c for c in chunks if 120 <= len(c) <= 1800]


def length_bucket(text: str) -> str:
    words = len(text.split())
    if words < 40:
        return "short"
    if words <= 120:
        return "medium"
    return "long"


def tags_for_text(text: str) -> list[str]:
    lowered = text.lower()
    tags: list[str] = []
    for tag, words in KEYWORD_TAGS.items():
        if any(word in lowered for word in words):
            tags.append(tag)
    return tags[:3] if tags else ["presence"]

In [ ]:
passages: list[dict] = []
for txt in sorted(GOLD_DIR.glob("*.txt")):
    if txt.name not in GOLD_SOURCES:
        continue
    tradition, source = GOLD_SOURCES[txt.name]
    raw = txt.read_text(encoding="utf-8", errors="ignore")
    for chunk in chunk_text(raw):
        passages.append(
            {
                "tradition": tradition,
                "source": source,
                "text": chunk,
                "tone": "gentle",  # current ingestion default; tone enrichment is on the parking lot
                "length_bucket": length_bucket(chunk),
                "theme_tags": tags_for_text(chunk),
                "source_tier": "gold",
            }
        )

print(f"Total gold passages chunked: {len(passages)}")
traditions_seen = sorted({p["tradition"] for p in passages})
print(f"Traditions: {len(traditions_seen)}")
for t in traditions_seen:
    print(f"  {t}")

## Passages per tradition

The corpus is intentionally uneven — Marcus Aurelius and Thoreau are loquacious, Tao Te Ching and the Bhagavad Gita are compressed by translator (and by the verse format). The bandit treats each tradition as part of a super-arm, so traditions with very few passages will be filtered out by the `min_count=3` rule before any sampling happens.


In [ ]:
tradition_counts = Counter(p["tradition"] for p in passages)
ordered = tradition_counts.most_common()
labels = [t for t, _ in ordered]
counts = [c for _, c in ordered]

print(f"Total: {sum(counts)} passages across {len(counts)} traditions.\n")

fig, ax = plt.subplots(figsize=(10, max(3.5, 0.32 * len(ordered) + 1.5)))
bars = ax.barh(labels, counts, color="#cb9366", edgecolor="#7a6e62", height=0.7)
ax.invert_yaxis()
ax.set_xlabel("number of passages")
ax.set_title("Gold-tier passages per tradition")
for bar, c in zip(bars, counts):
    ax.text(
        bar.get_width() + max(counts) * 0.005,
        bar.get_y() + bar.get_height() / 2,
        str(c),
        va="center",
        fontsize=10,
        color="#3a322c",
    )
ax.grid(True, axis="x", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Length distribution per tradition

The chunker buckets passages into `short` (< 40 words), `medium` (40-120), and `long` (> 120). The generator's style rotation has no preference today, but knowing the distribution per tradition helps catch outliers — e.g. a tradition that's almost entirely "long" passages will dominate when style D (nature/seasons) requests longer reflective excerpts.


In [ ]:
length_by_tradition: dict[str, Counter] = defaultdict(Counter)
for p in passages:
    length_by_tradition[p["tradition"]][p["length_bucket"]] += 1

bucket_order = ["short", "medium", "long"]
bucket_colors = {"short": "#dee5d3", "medium": "#cb9366", "long": "#2f5b4f"}

ordered_traditions = [t for t, _ in tradition_counts.most_common()]
fig, ax = plt.subplots(figsize=(10, max(3.5, 0.32 * len(ordered_traditions) + 1.5)))
left = np.zeros(len(ordered_traditions))
for bucket in bucket_order:
    values = [length_by_tradition[t].get(bucket, 0) for t in ordered_traditions]
    ax.barh(
        ordered_traditions,
        values,
        left=left,
        color=bucket_colors[bucket],
        edgecolor="#7a6e62",
        label=bucket,
        height=0.7,
    )
    left += np.array(values)
ax.invert_yaxis()
ax.set_xlabel("number of passages")
ax.set_title("Length-bucket distribution per tradition")
ax.legend(loc="lower right", frameon=False)
ax.grid(True, axis="x", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Theme-tag coverage

The keyword-based tagger in `seed_passages_to_supabase.py` is intentionally simple — it scans for the presence of theme keywords (e.g. "courage", "discipline", "nature") and assigns up to 3 tags per passage, defaulting to `presence` when nothing matches. This gives the retrieval layer something to filter on without a manual labeling pass.

A theme that no passage carries is a hole in the corpus — a query like "How do I sit with sorrow?" will silently fall back to plain embedding similarity. The bar chart below makes those holes visible.


In [ ]:
theme_counts: Counter = Counter()
for p in passages:
    for tag in p["theme_tags"]:
        theme_counts[tag] += 1

# Order by the canonical KEYWORD_TAGS keys plus any extras we found.
theme_order = list(KEYWORD_TAGS.keys()) + sorted(set(theme_counts) - set(KEYWORD_TAGS))
values = [theme_counts.get(t, 0) for t in theme_order]

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(theme_order, values, color="#2f5b4f", edgecolor="#7a6e62")
ax.set_title("Theme-tag coverage across the gold corpus")
ax.set_ylabel("passages tagged")
plt.xticks(rotation=35, ha="right")
ax.grid(True, axis="y", alpha=0.25)
ax.set_axisbelow(True)
for bar, v in zip(bars, values):
    if v > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.01,
            str(v),
            ha="center",
            va="bottom",
            fontsize=9,
            color="#3a322c",
        )
plt.tight_layout()
plt.show()

zero_tags = [t for t in theme_order if theme_counts.get(t, 0) == 0]
if zero_tags:
    print(f"\nThemes with zero passages: {zero_tags}")
else:
    print("\nEvery declared theme has at least one passage.")

## Bandit super-arm viability

Every `(tradition, tone)` pair with at least 3 passages is a *super-arm* the bandit can sample (see `MIN_PASSAGES_PER_ARM` in `services/bandit.py`). Pairs with fewer passages are silently filtered. The current ingestion pipeline assigns `tone = "gentle"` to every passage, so today the super-arm space collapses to one arm per tradition — but the column is in place so a future tone-enrichment pass (Phase B parking lot) can split arms without a schema change.


In [ ]:
MIN_PASSAGES_PER_ARM = 3

arm_counts: Counter = Counter()
for p in passages:
    arm_counts[(p["tradition"], p["tone"])] += 1

viable = [(arm, n) for arm, n in arm_counts.items() if n >= MIN_PASSAGES_PER_ARM]
filtered = [(arm, n) for arm, n in arm_counts.items() if n < MIN_PASSAGES_PER_ARM]

print(f"Distinct (tradition, tone) pairs:    {len(arm_counts)}")
print(f"  Viable super-arms (n >= {MIN_PASSAGES_PER_ARM}):     {len(viable)}")
print(f"  Filtered (would be excluded):     {len(filtered)}")
if filtered:
    for arm, n in sorted(filtered, key=lambda kv: kv[1]):
        print(f"     {arm[0]:<22} | {arm[1]:<8} | n={n}")
print("\nViable super-arms (these are what the bandit will actually sample):")
for arm, n in sorted(viable, key=lambda kv: -kv[1]):
    print(f"  {arm[0]:<22} | {arm[1]:<8} | n={n}")